In [ ]:
import os
import shutil
from google.colab import drive

# --- 1. Mount Drive ---
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# --- 2. Paths ---
PROJECT_DRIVE = '/content/drive/MyDrive/PlantDiseaseProject'
LOCAL_DATA = '/content/data'

# unzip locally
need_unzip = not (
    os.path.exists(os.path.join(LOCAL_DATA, 'PlantVillage')) and
    len(os.listdir(os.path.join(LOCAL_DATA, 'PlantVillage'))) > 30
)

if need_unzip:
    print("Re-unzipping data to local SSD...")
    for f in ['PlantVillage', 'PlantDoc']:
        path = os.path.join(LOCAL_DATA, f)
        if os.path.exists(path):
            shutil.rmtree(path)
    os.makedirs(os.path.join(LOCAL_DATA, 'PlantVillage'), exist_ok=True)
    os.makedirs(os.path.join(LOCAL_DATA, 'PlantDoc'), exist_ok=True)

    !unzip -q "$PROJECT_DRIVE/plantvillage-dataset.zip" -d "$LOCAL_DATA/PlantVillage"
    !unzip -q "$PROJECT_DRIVE/PlantDoc-Dataset-master.zip" -d "$LOCAL_DATA/PlantDoc"

    # Flatten PlantVillage
    pv_nested = os.path.join(LOCAL_DATA, 'PlantVillage', 'plantvillage dataset')
    if os.path.exists(pv_nested):
        color_path = os.path.join(pv_nested, 'color')
        source = color_path if os.path.exists(color_path) else pv_nested
        for item in os.listdir(source):
            shutil.move(os.path.join(source, item), os.path.join(LOCAL_DIR := LOCAL_DATA, 'PlantVillage', item))

    # Flatten PlantDoc
    pd_nested = os.path.join(LOCAL_DATA, 'PlantDoc', 'PlantDoc-Dataset-master')
    if os.path.exists(pd_nested):
        for item in os.listdir(pd_nested):
            shutil.move(os.path.join(pd_nested, item), os.path.join(LOCAL_DATA, 'PlantDoc', item))
    print("Done unzipping!")
else:
    print("✅ Data already present locally.")

# --- 4. Verify split CSVs exist on Drive ---
split_dir = os.path.join(PROJECT_DRIVE, 'results', 'splits')
print(f"\nSplits on Drive: {os.listdir(split_dir)}")

# --- 5. Verify model/results folders exist ---
for sub in ['metrics', 'figures', 'models']:
    path = os.path.join(PROJECT_DRIVE, 'results', sub)
    os.makedirs(path, exist_ok=True)

print(f"\n✅ PlantVillage classes: {len(os.listdir(os.path.join(LOCAL_DATA, 'PlantVillage')))}")
print(f"✅ PlantDoc classes:     {len(os.listdir(os.path.join(LOCAL_DATA, 'PlantDoc', 'train')))}")

Mounted at /content/drive
Re-unzipping data to local SSD...
Done unzipping!

Splits on Drive: ['class_names.csv', 'train.csv', 'val.csv', 'test.csv']

✅ PlantVillage classes: 39
✅ PlantDoc classes:     28


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

PROJECT_DRIVE = '/content/drive/MyDrive/PlantDiseaseProject'
SPLIT_DIR = os.path.join(PROJECT_DRIVE, 'results', 'splits')

IMG_SIZE = 224
BATCH_SIZE = 32

def load_split_from_drive():
    train_df = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv'))
    val_df   = pd.read_csv(os.path.join(SPLIT_DIR, 'val.csv'))
    test_df  = pd.read_csv(os.path.join(SPLIT_DIR, 'test.csv'))
    class_df = pd.read_csv(os.path.join(SPLIT_DIR, 'class_names.csv'))
    class_to_idx = dict(zip(class_df['class'], class_df['index']))
    return train_df, val_df, test_df, class_to_idx

def decode_img(filepath, label, num_classes):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, num_classes)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.12)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(df, class_to_idx, num_classes, training=False):
    paths = df['filepath'].values
    labels = np.array([class_to_idx[l] for l in df['label'].values], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=2000, seed=42)
    ds = ds.map(lambda p, l: decode_img(p, l, num_classes),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def get_all_datasets():
    train_df, val_df, test_df, class_to_idx = load_split_from_drive()
    num_classes = len(class_to_idx)
    train_ds = make_dataset(train_df, class_to_idx, num_classes, training=True)
    val_ds   = make_dataset(val_df,   class_to_idx, num_classes, training=False)
    test_ds  = make_dataset(test_df,  class_to_idx, num_classes, training=False)
    return train_ds, val_ds, test_ds, class_to_idx

# Load them now so subsequent cells can use them
train_ds, val_ds, test_ds, class_to_idx = get_all_datasets()
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in range(len(class_to_idx))]
NUM_CLASSES = len(class_to_idx)

print(f"✅ Train batches: {len(train_ds)}")
print(f"✅ Val batches:   {len(val_ds)}")
print(f"✅ Test batches:  {len(test_ds)}")
print(f"✅ Classes:       {NUM_CLASSES}")

✅ Train batches: 1358
✅ Val batches:   170
✅ Test batches:  170
✅ Classes:       38


In [ ]:
import os

LOCAL_DATA = '/content/data'

# ================================================================
# MANUAL MAPPING: PlantDoc class (folder name) -> PlantVillage class
# ================================================================
PLANTDOC_TO_PLANTVILLAGE = {
    # Apple
    'Apple Scab Leaf':              'Apple___Apple_scab',
    'Apple leaf':                   'Apple___healthy',
    'Apple rust leaf':              'Apple___Cedar_apple_rust',

    # Bell pepper
    'Bell_pepper leaf':             'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':        'Pepper,_bell___Bacterial_spot',

    # Cherry
    'Cherry leaf':                  'Cherry_(including_sour)___healthy',

    # Corn
    'Corn Gray leaf spot':          'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':             'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf':               'Corn_(maize)___Common_rust_',

    # Grape
    'grape leaf':                   'Grape___healthy',
    'grape leaf black rot':         'Grape___Black_rot',

    # Peach
    'Peach leaf':                   'Peach___healthy',

    # Potato
    'Potato leaf early blight':     'Potato___Early_blight',
    'Potato leaf late blight':      'Potato___Late_blight',

    # Squash
    'Squash Powdery mildew leaf':   'Squash___Powdery_mildew',

    # Strawberry
    'Strawberry leaf':              'Strawberry___healthy',

    # Tomato
    'Tomato Early blight leaf':     'Tomato___Early_blight',
    'Tomato Septoria leaf spot':    'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                  'Tomato___healthy',
    'Tomato leaf bacterial spot':   'Tomato___Bacterial_spot',
    'Tomato leaf late blight':      'Tomato___Late_blight',
    'Tomato leaf mosaic virus':     'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':     'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':             'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
}

# --- Check which PlantDoc classes we have in the dataset ---
pd_test_dir = os.path.join(LOCAL_DATA, 'PlantDoc', 'test')
pd_classes = sorted([d for d in os.listdir(pd_test_dir)
                     if os.path.isdir(os.path.join(pd_test_dir, d))])

print(f"PlantDoc test classes ({len(pd_classes)}):")
for c in pd_classes:
    mapped = PLANTDOC_TO_PLANTVILLAGE.get(c, '❌ NO MAPPING')
    print(f"  {c:<45} → {mapped}")

# Count unmatched
unmatched = [c for c in pd_classes if c not in PLANTDOC_TO_PLANTVILLAGE]
matched = [c for c in pd_classes if c in PLANTDOC_TO_PLANTVILLAGE]

print(f"\n✅ Mapped:   {len(matched)} classes")
if unmatched:
    print(f"⚠️ Unmapped: {len(unmatched)} classes — will be excluded from evaluation:")
    for c in unmatched:
        print(f"     - {c}")

PlantDoc test classes (27):
  Apple Scab Leaf                               → Apple___Apple_scab
  Apple leaf                                    → Apple___healthy
  Apple rust leaf                               → Apple___Cedar_apple_rust
  Bell_pepper leaf                              → Pepper,_bell___healthy
  Bell_pepper leaf spot                         → Pepper,_bell___Bacterial_spot
  Blueberry leaf                                → ❌ NO MAPPING
  Cherry leaf                                   → Cherry_(including_sour)___healthy
  Corn Gray leaf spot                           → Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  Corn leaf blight                              → Corn_(maize)___Northern_Leaf_Blight
  Corn rust leaf                                → Corn_(maize)___Common_rust_
  Peach leaf                                    → Peach___healthy
  Potato leaf early blight                      → Potato___Early_blight
  Potato leaf late blight                       → Potato___

In [ ]:
# Add the missing healthy-class mappings
PLANTDOC_TO_PLANTVILLAGE.update({
    'Blueberry leaf':   'Blueberry___healthy',
    'Raspberry leaf':   'Raspberry___healthy',
    'Soyabean leaf':    'Soybean___healthy',   # Note: PlantVillage uses "Soybean"
})

# Verify all 27 classes now map
pd_test_dir = os.path.join(LOCAL_DATA, 'PlantDoc', 'test')
pd_classes = sorted([d for d in os.listdir(pd_test_dir)
                     if os.path.isdir(os.path.join(pd_test_dir, d))])

unmatched = [c for c in pd_classes if c not in PLANTDOC_TO_PLANTVILLAGE]

print(f"Mapped: {len(pd_classes) - len(unmatched)} / {len(pd_classes)} classes")
if unmatched:
    print(f"⚠️ Still unmapped: {unmatched}")
else:
    print("✅ All PlantDoc classes are now mapped!")

Mapped: 27 / 27 classes
✅ All PlantDoc classes are now mapped!


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf

LOCAL_DIR = '/content/data'
PROJECT_DIR = '/content/drive/MyDrive/PlantDiseaseProject'
SPLIT_DIR = os.path.join(PROJECT_DIR, 'results', 'splits')
IMG_SIZE = 224
BATCH_SIZE = 32

# Load class names from PlantVillage
class_df = pd.read_csv(os.path.join(SPLIT_DIR, 'class_names.csv'))
class_to_idx = dict(zip(class_df['class'], class_df['index']))

# Build a DataFrame of PlantDoc test images
pd_test_dir = os.path.join(LOCAL_DIR, 'PlantDoc', 'test')

rows = []
excluded_classes = []
for pd_class in sorted(os.listdir(pd_test_dir)):
    pd_class_path = os.path.join(pd_test_dir, pd_class)
    if not os.path.isdir(pd_class_path):
        continue

    if pd_class not in PLANTDOC_TO_PLANTVILLAGE:
        excluded_classes.append(pd_class)
        continue

    pv_class = PLANTDOC_TO_PLANTVILLAGE[pd_class]
    if pv_class not in class_to_idx:
        print(f"⚠️ {pd_class} maps to {pv_class} which is not in PlantVillage classes")
        continue

    for img in os.listdir(pd_class_path):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append({
                'filepath': os.path.join(pd_class_path, img),
                'label': class_to_idx[pv_class],
                'original_label': pd_class,
            })

plantdoc_df = pd.DataFrame(rows)
print(f"PlantDoc test images (mapped): {len(plantdoc_df):,}")
print(f"Classes represented: {plantdoc_df['original_label'].nunique()}")
if excluded_classes:
    print(f"Excluded classes: {excluded_classes}")

# Build tf.data.Dataset
def decode_img_plantdoc(filepath, label, num_classes):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, num_classes)
    return img, label

def make_plantdoc_dataset(df, num_classes):
    paths = df['filepath'].values
    labels = df['label'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: decode_img_plantdoc(p, l, num_classes),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

NUM_CLASSES = len(class_to_idx)
plantdoc_ds = make_plantdoc_dataset(plantdoc_df, NUM_CLASSES)

print(f"\n✅ PlantDoc dataset built — {len(plantdoc_ds)} batches")

PlantDoc test images (mapped): 236
Classes represented: 27

✅ PlantDoc dataset built — 8 batches


In [ ]:
import os, json
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/PlantDiseaseProject'
METRICS_DIR = os.path.join(PROJECT_DIR, 'results', 'metrics')
SPLIT_DIR   = os.path.join(PROJECT_DIR, 'results', 'splits')
MODELS_DIR  = os.path.join(PROJECT_DIR, 'results', 'models')
IMG_SIZE, BATCH_SIZE = 224, 32

tf.keras.config.enable_unsafe_deserialization()

print("Loading EfficientNetB0...")
model = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, 'efficientnetb0.keras'), safe_mode=False)

for layer in model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

class_df = pd.read_csv(os.path.join(SPLIT_DIR, 'class_names.csv'))
class_to_idx = dict(zip(class_df['class'], class_df['index']))

pd_test_dir = '/content/data/PlantDoc/test'
rows = []
for cls in sorted(os.listdir(pd_test_dir)):
    p = os.path.join(pd_test_dir, cls)
    if not os.path.isdir(p): continue
    if cls not in PLANTDOC_TO_PLANTVILLAGE: continue
    pv = PLANTDOC_TO_PLANTVILLAGE[cls]
    if pv not in class_to_idx: continue
    for img in os.listdir(p):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append({'path': os.path.join(p, img),
                         'label': class_to_idx[pv]})

df = pd.DataFrame(rows)
ft_rows, eval_rows = [], []
for lab, grp in df.groupby('label'):
    grp = grp.sample(frac=1, random_state=42)
    ft_rows.append(grp.head(5))
    eval_rows.append(grp.iloc[5:])

ft_df   = pd.concat(ft_rows).reset_index(drop=True)
eval_df = pd.concat(eval_rows).reset_index(drop=True)

print(f"Fine-tune set: {len(ft_df)} images | Eval set: {len(eval_df)} images")

NUM_CLASSES = len(class_to_idx)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.one_hot(label, NUM_CLASSES)

def make_ds(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['path'].values, df['label'].values.astype(np.int32)))
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ft_ds   = make_ds(ft_df, shuffle=True)
eval_ds = make_ds(eval_df)

print("\nFine-tuning on 5 PlantDoc images/class...")
history = model.fit(ft_ds, epochs=10, verbose=1)

y_pred_probs = model.predict(eval_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in eval_ds])
ft_acc = accuracy_score(y_true, y_pred)

print(f"\n{'='*60}")
print(f"Fine-tuned EfficientNetB0 on PlantDoc eval set: {ft_acc*100:.2f}%")
print(f"Zero-shot baseline (Cell 4):                    25.85%")
print(f"Improvement:                                    +{(ft_acc-0.2585)*100:.2f} pp")
print(f"{'='*60}")

with open(os.path.join(METRICS_DIR, 'plantdoc_finetuned_efficientnet.json'), 'w') as f:
    json.dump({
        'finetuned_accuracy': float(ft_acc),
        'baseline_accuracy': 0.2585,
        'improvement_pp': float((ft_acc - 0.2585) * 100),
        'ft_samples': len(ft_df),
        'eval_samples': len(eval_df),
    }, f, indent=2)

print(f"\n✅ Saved: {METRICS_DIR}/plantdoc_finetuned_efficientnet.json")

Loading EfficientNetB0...
Fine-tune set: 134 images | Eval set: 102 images

Fine-tuning on 5 PlantDoc images/class...
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 150s 15s/step - accuracy: 0.1194 - loss: 9.7921
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 120ms/step - accuracy: 0.1343 - loss: 6.6905
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.2687 - loss: 4.7105
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 120ms/step - accuracy: 0.3433 - loss: 3.5502
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.3955 - loss: 2.6205
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 119ms/step - accuracy: 0.5000 - loss: 2.4411
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - accuracy: 0.5746 - loss: 1.6388
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step - accuracy: 0.6418 - loss: 1.5209
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 122ms/step - accuracy: 0.6866 - loss: 1.0816
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 122ms/step - accuracy: 0.7761 - loss: 0.8538

Fine-tuned EfficientNetB0 on PlantDoc ev